# USB dan HID — enumerasi, control, bulk, report

Transfer USB mentah dan report HID lewat satu API. Sel-selnya memakai bus virtual; ganti backend dengan `NativeUsbBackend.Instance` (bawaan) untuk menjangkau perangkat sungguhan.

## Persiapan
> Bekerja dari hasil clone? Jalankan `dotnet pack -c Release -o artifacts/packages` di root repo lalu tambahkan
> `#i "nuget: <repo>/artifacts/packages"` sebelum baris `#r`.

In [ ]:
#r "nuget: IoTCom.Net, 0.21.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.21.0-preview.1"

## Enumerasi

In [ ]:
using System.Text;
using IoTCom.Net.Transport.Usb;

var usbBus = new VirtualUsbBus();
usbBus.Add(new VirtualLoopbackDevice());
var boardSim = usbBus.AddHid(new VirtualHidRelayBoard(relays: 4));
foreach (var d in UsbDevice.List(usbBus)) Console.WriteLine($"{d.Id}  {d.Manufacturer} {d.Product}  [{string.Join(", ", d.Interfaces.Select(i => i.ClassName))}]");
foreach (var h in HidDevice.List(usbBus)) Console.WriteLine($"{h.VendorId:x4}:{h.ProductId:x4}  {h.Product}  {h.UsageName}");
Console.WriteLine(UsbIds.Known(0x1D50, 0x606F));

## Transfer control dan bulk
Vendor request membaca versi firmware; data yang ditulis ke bulk OUT 0x01 kembali di bulk IN 0x81.

In [ ]:
var loopDev = UsbDevice.Create(o => { o.UseVirtual(usbBus); o.DeviceId = "1209:0001"; });
await loopDev.ConnectAsync();
Console.WriteLine(Encoding.ASCII.GetString(await loopDev.ControlInAsync(UsbSetup.Vendor(0x01), 16)));
await loopDev.WriteAsync(0x01, Encoding.ASCII.GetBytes("ping"));
Console.WriteLine(Encoding.ASCII.GetString((await loopDev.ReadAsync(0x81))!));
await loopDev.DisposeAsync();

## Papan relay USB HID
Feature report `00 FF n` menyalakan relay n; membaca feature report 0 mengembalikan serial dan bit relay.

In [ ]:
var relayHid = HidDevice.Create(o => { o.UseVirtual(usbBus); o.UseDevice(HidRelayBoard.VendorId, HidRelayBoard.ProductId); });
await relayHid.ConnectAsync();
var relayBoard = new HidRelayBoard(relayHid);
await relayBoard.SetAsync(3, true);
Console.WriteLine($"{await relayBoard.GetSerialAsync()}: {string.Join(" ", await relayBoard.GetStatesAsync())} (bits {boardSim.State})");
var readOnlyHid = HidDevice.Create(o => { o.UseVirtual(usbBus); o.UseDevice(HidRelayBoard.VendorId, HidRelayBoard.ProductId); o.ReadOnly = true; });
await readOnlyHid.ConnectAsync();
try { await new HidRelayBoard(readOnlyHid).SetAllAsync(true); } catch (IoTCom.Net.ReadOnlyModeException e) { Console.WriteLine("read-only: " + e.Message); }

## Lebih lanjut
`iotcom usb list`, `iotcom usb hid`, `iotcom usb relay --sim`, *Meja kerja USB* di Gallery, dan sampel UsbRelay. Lihat `docs/id/protocols/usb.md` untuk driver dan izin.

*IoTCom.Net — dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*